In [0]:
spark.sql(""" select * from dev.spark_db.bookings""").limit(2).display()

In [0]:
spark.sql(""" select * from dev.spark_db.facilities""").limit(2).display()

In [0]:
spark.sql(""" select * from dev.spark_db.members""").limit(2).display()

In [0]:
spark.sql("""
          
          select f.member_id , f.first_name , f.last_name , b.facility_id , b.slots from dev.spark_db.members f
          inner join dev.spark_db.bookings b
          on f.member_id = b.member_id
          where b.slots > 5 and last_name = 'Smith'
          order by f.first_name asc , b.slots desc       
          """).display()

In [0]:

from pyspark.sql.functions import col
spark.sql("""          
          select f.member_id , f.first_name , f.last_name , b.facility_id , b.slots , b.start_time from dev.spark_db.members f
          inner join dev.spark_db.bookings b
          on f.member_id = b.member_id      
          """).where( (col("last_name") == 'Smith') & (col("slots") > 5 ))\
              .orderBy(col("first_name").asc() , col("slots").desc())\
                  .display()

In [0]:

members_df = spark.sql(""" select * from dev.spark_db.members""")
bookings_df = spark.sql("""select * from dev.spark_db.bookings""") 

In [0]:
members_df.join(bookings_df , on = members_df.member_id == bookings_df.member_id , how = 'inner')\
            .where( (col("last_name") == 'Smith') & (col("slots") > 5))\
                .orderBy(col("first_name").asc() , col("slots").desc())\
                    .select(members_df.member_id , "last_name" , "first_name" , "slots" , "start_time")\
                    .display()

In [0]:
members_df.join(bookings_df , on = members_df.member_id == bookings_df.member_id , how = 'inner').display()

In [0]:
bookings_df = spark.table("dev.spark_db.bookings").alias("b")
members_df = spark.table("dev.spark_db.members").alias("m")
facilities_df = spark.table("dev.spark_db.facilities").alias("f")

In [0]:
from pyspark.sql.functions import expr

join_expr = col("m.member_id") == col("b.member_id")
# join_expr = expr("m.member_id == b.member_id")

members_df.join(bookings_df , join_expr , 'inner')\
            .where("m.last_name = 'Smith' and  b.slots > 5")\
                .select("m.member_id" , "last_name" , "first_name" , "slots" , "start_time")\
                    .orderBy(expr(" m.first_name  asc") ,col("slots").desc())\   
                        .display()

                        #orderBy(expr( desc ))  -- it is not working glich

In [0]:
members_df.join(bookings_df , expr("m.member_id == b.member_id") , 'inner')\
          .join(facilities_df , bookings_df.facility_id == facilities_df.facility_id  ,'inner' )\
          .where( (members_df.last_name == "Smith") & (bookings_df.slots > 5))\
          .select("m.first_name" , "m.last_name" , expr("b.slots * f.member_cost as booking_cost"))\
          .orderBy(col("first_name").asc() , col("slots").desc())\
          .display()
